# いろどりTTS 全自動生成

### 手順
1. **セル1▶** いろどりTTSをインストール（約10分、待つだけ）
2. **セル2▶** blocks.txtと参照音声をアップロード
3. **セル3▶** 全ブロック自動生成（放置でOK）
4. **セル4▶** zipでダウンロード

> ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択してから実行

In [ ]:
# セル1：インストール（約10分）
import os

LOCAL_DIR = '/content/Irodori-TTS'

if os.path.exists(LOCAL_DIR):
    print('インストール済み、スキップします')
else:
    print('インストール中...（約10分）')
    !git clone https://github.com/Aratako/Irodori-TTS.git {LOCAL_DIR}
    !pip install -q uv
    !cd {LOCAL_DIR} && uv sync
    print('インストール完了！')

os.chdir(LOCAL_DIR)
print('準備完了')
!nvidia-smi | head -3

In [ ]:
# セル2：blocks.txtと参照音声をアップロード
from google.colab import files
import os

print('blocks.txt と 参照音声(.wav) を選択してください')
uploaded = files.upload()

blocks_file = None
ref_wav = None

for filename in uploaded:
    if filename.endswith('.txt'):
        blocks_file = '/content/Irodori-TTS/blocks.txt'
        with open(blocks_file, 'wb') as f:
            f.write(uploaded[filename])
        print(f'blocks.txt: OK')
    elif filename.endswith('.wav'):
        os.makedirs('/content/Irodori-TTS/ref', exist_ok=True)
        ref_wav = '/content/Irodori-TTS/ref/reference.wav'
        with open(ref_wav, 'wb') as f:
            f.write(uploaded[filename])
        print(f'参照音声: OK')

if blocks_file:
    with open(blocks_file, 'r', encoding='utf-8') as f:
        content = f.read()
    blocks = [b.strip() for b in content.split('\n\n') if b.strip()]
    print(f'\nブロック数: {len(blocks)}')
    print(f'参照音声: {ref_wav or "なし"}')
else:
    print('blocks.txt がアップロードされていません')

In [ ]:
# セル3：全ブロック自動生成
import subprocess, time, os
from pathlib import Path
from datetime import datetime

LOCAL_DIR = '/content/Irodori-TTS'
HF_CHECKPOINT = 'Aratako/Irodori-TTS-500M-v2'
SEED = 42

def normalize_text(text):
    text = text.replace('\u300c', '').replace('\u300d', '')
    text = text.replace('\u300e', '').replace('\u300f', '')
    text = text.replace('\u2026\u2026', '\u3002').replace('\u2026', '\u3002')
    text = text.replace('\u30fb\u30fb\u30fb', '\u3002')
    text = text.replace('\uff1f', '?').replace('\uff01', '!').replace('\uff5e', '~')
    text = text.replace('\u2015\u2015', '\u3002').replace('\u2015', '\u3001')
    stripped = text.rstrip()
    if stripped and stripped[-1] not in '\u3002\u3001!?~.':
        text = stripped + '\u3002'
    return text

def generate_one(text, output_wav, ref_wav=None, seed=42):
    text = normalize_text(text)
    cmd = ['uv', 'run', 'python', 'infer.py',
           '--hf-checkpoint', HF_CHECKPOINT,
           '--text', text,
           '--output-wav', output_wav,
           '--seed', str(seed),
           '--num-steps', '80',
           '--num-candidates', '1']
    if ref_wav and os.path.exists(ref_wav):
        cmd += ['--ref-wav', ref_wav]
    else:
        cmd += ['--no-ref']
    result = subprocess.run(cmd, cwd=LOCAL_DIR, capture_output=True, text=True, encoding='utf-8', errors='replace')
    if result.returncode != 0:
        print(f'ERR: {result.stderr[-200:]}')
    return result.returncode == 0

with open(blocks_file, 'r', encoding='utf-8') as f:
    content = f.read()
blocks = [b.strip() for b in content.split('\n\n') if b.strip()]
total = len(blocks)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_dir = Path(f'{LOCAL_DIR}/gradio_outputs/{timestamp}')
out_dir.mkdir(parents=True, exist_ok=True)

print(f'=== いろどりTTS 全自動生成 ===')
print(f'ブロック数 : {total}')
print(f'参照音声   : {ref_wav or "なし"}')
print(f'出力先     : {out_dir}')
print()

failed = []
start_time = time.time()

for i, block in enumerate(blocks):
    block_num = i + 1
    out_wav = str(out_dir / f'block_{block_num:04d}.wav')
    if os.path.exists(out_wav):
        print(f'[{block_num:03d}/{total}] スキップ（既存）')
        continue
    preview = block[:35].replace('\n', ' ')
    print(f'[{block_num:03d}/{total}] 「{preview}...」', end=' ', flush=True)
    t0 = time.time()
    success = generate_one(block, out_wav, ref_wav, SEED)
    elapsed = time.time() - t0
    if success:
        print(f'✓ ({elapsed:.1f}秒)')
    else:
        print(f'✗ 失敗')
        failed.append(block_num)

total_time = time.time() - start_time
print(f'\n=== 完了 ===')
print(f'成功: {total - len(failed)} / {total}')
print(f'失敗: {failed if failed else "なし"}')
print(f'合計: {total_time/60:.1f}分')
print(f'出力先: {out_dir}')

In [ ]:
# セル4：zipでダウンロード
import zipfile
from google.colab import files
from pathlib import Path

wav_files = sorted(out_dir.glob('*.wav'))
zip_path = f'/content/output_{timestamp}.zip'

print(f'{len(wav_files)}個のwavをzip化中...')
with zipfile.ZipFile(zip_path, 'w') as zf:
    for wav in wav_files:
        zf.write(wav, wav.name)

print('ダウンロード開始...')
files.download(zip_path)